# 🏓 Beer Pong Trajectory Analysis

This notebook turns the tracker output into sports-analytics insights:

1. Load a trajectory (`data/trajectory.csv` from `src/track_ball.py`) — or fall back to **synthetic throws** so it runs out of the box
2. Segment flights into shots
3. Fit ballistic parabolas and compute launch angle, apex and flight time
4. Classify **hit vs. miss** against the cup rack
5. Visualise trajectories, landing spots and accuracy

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.trajectory import (default_cup_rack, fit_parabola, load_cups,
                            split_shots, summarize, synthetic_throws)

plt.rcParams.update({"figure.figsize": (11, 5), "axes.grid": True, "grid.alpha": 0.3})

## 1 · Load data

In [ ]:
TRAJ_CSV = ROOT / "data" / "trajectory.csv"
CUPS_JSON = ROOT / "data" / "cups.json"

if TRAJ_CSV.exists():
    raw = pd.read_csv(TRAJ_CSV)
    print(f"Loaded {len(raw)} detections from {TRAJ_CSV.name}")
else:
    raw = synthetic_throws(n_shots=10)
    print("No tracker output found -> using synthetic demo throws")

cups = load_cups(CUPS_JSON) if CUPS_JSON.exists() else default_cup_rack()
raw.head()

## 2 · Segment shots & compute metrics

In [ ]:
df = split_shots(raw.drop(columns=["shot"], errors="ignore"), max_gap=8)
summary = summarize(df, cups, px_per_meter=None)
summary.round(2)

## 3 · Trajectories in image space
Image y grows downward, so the axis is inverted to show the arc the right way up.

In [ ]:
fig, ax = plt.subplots()
colors = {"hit": "tab:green", "miss": "tab:red"}
for _, row in summary.iterrows():
    shot = df[df["shot"] == row["shot"]]
    ax.plot(shot["x"], shot["y"], "-", lw=1.5, alpha=0.8, color=colors.get(row.get("outcome"), "tab:blue"))
    ax.plot(row["landing_x"], row["landing_y"], "x", ms=9, color=colors.get(row.get("outcome"), "k"))
for c in cups:
    ax.add_patch(plt.Circle((c.x, c.y), c.r, fill=False, ec="0.3", lw=1.2))
ax.invert_yaxis(); ax.set_aspect("equal")
ax.set(title="Ball trajectories (green = hit, red = miss)", xlabel="x [px]", ylabel="y [px]")
plt.show()

## 4 · Ballistic fit quality
A clean throw follows $y(t) = a t^2 + b t + c$. Large residuals hint at tracking errors or a bounce.

In [ ]:
shot0 = df[df["shot"] == 0]
a, b, c = fit_parabola(shot0)
t = shot0["t"] - shot0["t"].iloc[0]
tt = np.linspace(0, t.max(), 200)
fig, ax = plt.subplots()
ax.scatter(t, shot0["y"], s=12, label="tracked")
ax.plot(tt, np.polyval([a, b, c], tt), "r-", label=f"fit  a={a:.0f} px/s²")
ax.invert_yaxis(); ax.legend()
ax.set(title="Shot 0 — vertical position vs. time", xlabel="t [s]", ylabel="y [px]")
plt.show()

## 5 · What separates hits from misses?

In [ ]:
if "outcome" in summary:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for ax, col in zip(axes, ["launch_angle_deg", "launch_speed_px_s", "flight_time_s"]):
        for outcome, grp in summary.groupby("outcome"):
            ax.hist(grp[col], bins=8, alpha=0.6, label=outcome, color=colors[outcome])
        ax.set_title(col); ax.legend()
    plt.tight_layout(); plt.show()

    rate = (summary["outcome"] == "hit").mean()
    print(f"Hit rate: {rate:.0%}  ({(summary['outcome'] == 'hit').sum()}/{len(summary)})")
    display(summary.groupby("outcome")[["launch_angle_deg", "launch_speed_px_s", "apex_y", "flight_time_s"]].mean().round(2))

## Next steps
- Calibrate `px_per_meter` (cup diameter ≈ 9.5 cm) for real-world speeds
- Collect a few dozen throws per player and compare release consistency
- Add pose features (release height, elbow angle) from `src/beerpong_science.py` / MediaPipe